# Risk Rectification in Heteroscedastic Regression — Route 1 with TabPFN Quantile Sampling

This notebook recreates Experiment 1 but replaces **M3) Rectified Route 2** with **M3) Rectified Route 1**.

The three methods compared are:

- **M1) Standard CRC**: one global value of $\lambda$ calibrated by CRC.
- **M2) Oracle Rectified CRC**: $R(\lambda \mid x)$ is computed from the known Gaussian DGP.
- **M3) Rectified Route 1**: estimate a predictive distribution $\hat p_\phi(y \mid x, D)$ using a TabPFN tabular model, draw Monte Carlo samples $Y^{(s)}(x)$ from predicted quantiles, estimate $R(\lambda \mid x)$ by averaging the loss over the samples, then apply rectified CRC.

The base prediction band is still centered at a GAM estimate $\widehat\mu(x)$:

$$
C_\lambda(x)=\left[\widehat\mu(x)-\lambda(x),\widehat\mu(x)+\lambda(x)\right].
$$

The loss remains the bounded absolute excess loss:

$$
L_\lambda(x,y)=\min\left\{1, \frac{(|y-\widehat\mu(x)|-\lambda(x))_+}{B_{\mathrm{excess}}}\right\}.
$$

Route 1 implementation detail: TabPFN supplies estimated conditional quantiles. The code samples random probability levels $u\in[0.01,0.99]$, interpolates the predicted quantile function $\widehat Q(u\mid x)$, and treats the interpolated values as Monte Carlo draws from $\hat p_\phi(y\mid x,D)$.

The true DGP is used only for data generation, the oracle benchmark, and diagnostic difficulty bins.


## Environment setup

This machine blocks system-wide `pip install` calls because Python is externally managed. Create a project virtual environment from the terminal, then select that kernel in Jupyter:

```bash
python3 -m venv .venv
. .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements-route1.txt
python -m ipykernel install --user --name crc-route1 --display-name "CRC Route 1 (.venv)"
```

After that, restart this notebook and choose the kernel **CRC Route 1 (.venv)**.


In [ ]:
# Environment check
import sys

required = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scipy",
    "sklearn",
    "pygam",
    "tabpfn",
]

missing = []
for package in required:
    try:
        __import__(package)
    except ImportError:
        missing.append(package)

if missing:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing) +
        "\nCreate/select the project virtual environment described above before running the notebook."
    )

try:
    import torch
    print("CUDA disponível:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch is not installed yet; TabPFN may install it as a dependency depending on your environment.")

print("Python executable:", sys.executable)


## TabPFN license token

TabPFN requires a one-time Prior Labs license/API token before it can download model weights. If the token is not already set in your environment, run the next cell and paste the API key from https://ux.priorlabs.ai/account. The input is hidden and is only stored in the current notebook kernel environment.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("TABPFN_TOKEN"):
    os.environ["TABPFN_TOKEN"] = getpass("Paste TABPFN_TOKEN / Prior Labs API key: ")
else:
    print("TABPFN_TOKEN is already set.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pygam import LinearGAM, s
from tabpfn import TabPFNRegressor
from dataclasses import dataclass
import time

sns.set_theme(style="whitegrid", context="notebook")


## 1. DGP


In [ ]:
def mu_true(x):
    return np.sin(np.pi * x / 2)


def sigma_true(x):
    return 0.2 + 0.6 * np.abs(x)


def gerar_dataset(n, rng):
    X = rng.uniform(-2, 2, size=n)
    mu = mu_true(X)
    sg = sigma_true(X)
    Y = mu + sg * rng.standard_normal(n)
    return pd.DataFrame({"X": X, "Y": Y, "mu_true": mu, "sigma_true": sg})


## 2. Base regression model $\widehat\mu(x)$

The GAM is used only to center the prediction bands.


In [ ]:
def fit_mu(D: pd.DataFrame, n_splines=20, lam=0.6):
    """Fit a univariate GAM for E[Y | X]."""
    gam = LinearGAM(s(0, n_splines=n_splines), lam=lam)
    gam.fit(D[["X"]].values, D["Y"].values)
    return gam


def predict_mu(gam, x):
    """Predict the conditional mean using a fitted GAM."""
    x = np.atleast_1d(x).reshape(-1, 1)
    return gam.predict(x)


## 3. Loss and oracle risk


In [ ]:
B_LOSS = 1.0
EXCESS_SCALE = 1.5


def bounded_excess_loss(residual, lam, excess_scale=EXCESS_SCALE):
    """min(1, max(0, residual - lambda) / B_excess)."""
    return np.minimum(
        1.0,
        np.maximum(0.0, residual - lam) / excess_scale
    )


def oracle_R_dist_unbounded(lam, sigma, bias=0.0):
    """
    E[max(0, |W| - lam)] with W = Y - mu_hat ~ N(-bias, sigma^2),
    where bias = mu_hat(x) - mu_true(x).
    """
    lam_p = lam - bias
    lam_m = lam + bias
    z_p = lam_p / sigma
    z_m = lam_m / sigma

    return (
        sigma * (stats.norm.pdf(z_p) + stats.norm.pdf(z_m))
        - lam * (stats.norm.cdf(-z_p) + stats.norm.cdf(-z_m))
        + bias * (stats.norm.cdf(-z_p) - stats.norm.cdf(-z_m))
    )


def oracle_R_dist_bounded(lam, sigma, bias=0.0, excess_scale=EXCESS_SCALE):
    """
    Oracle conditional risk for E[min(1, (|W|-lam)_+ / B_excess)].
    """
    return (
        oracle_R_dist_unbounded(lam, sigma, bias)
        - oracle_R_dist_unbounded(lam + excess_scale, sigma, bias)
    ) / excess_scale


def oracle_R_matriz(x_vec, lambda_grid, mu_hat_fn):
    """Matrix n x L with oracle R(lambda_j | x_i)."""
    sg = sigma_true(x_vec)
    bias = mu_hat_fn(x_vec) - mu_true(x_vec)

    out = np.empty((len(x_vec), len(lambda_grid)))
    for j, lam in enumerate(lambda_grid):
        out[:, j] = oracle_R_dist_bounded(lam, sg, bias)

    return np.clip(out, 0.0, B_LOSS)


## 4. Rectified Route 1 — estimate $Y \mid X$, then average the loss

This replaces Experiment 1's Route 2 risk-surface regression.

Instead of directly fitting $(X,\lambda)\mapsto R(\lambda\mid X)$, Route 1 first fits an estimated predictive distribution for $Y\mid X$. Then, for every calibration/test input $x_i$:

1. draw $S$ samples $Y_i^{(1)},\ldots,Y_i^{(S)}$ from the TabPFN quantile model;
2. compute residuals $|Y_i^{(s)}-\widehat\mu(x_i)|$;
3. evaluate the bounded loss for every $\lambda_m$;
4. average over samples to estimate the local risk curve.

The same Monte Carlo samples are reused for all $\lambda_m$ at a given $x_i$, so the loss curve is already monotone in $\lambda$ up to numerical precision. No isotonic regression step is applied.


In [ ]:
class TabPFNExtractor:
    """
    Small wrapper around TabPFNRegressor for conditional quantile sampling.

    It follows the old-script pattern:
        fm = TabPFNExtractor(n_estimators=8, device="cuda")
        fm.fit(X_train, y_train)
        fm.model_.predict(X_new, output_type="quantiles", quantiles=grid)
    """

    def __init__(self, n_estimators=8, **kwargs):
        self.n_estimators = n_estimators
        self.kwargs = kwargs
        self.model_ = None

    def fit(self, X, y):
        self.model_ = TabPFNRegressor(
            n_estimators=self.n_estimators,
            **self.kwargs
        )
        self.model_.fit(X, y)
        return self

    def predict_quantiles(self, X, quantiles):
        q = self.model_.predict(
            X,
            output_type="quantiles",
            quantiles=list(quantiles)
        )
        return np.stack(q, axis=1)


def fit_y_given_x_sampler_tabpfn(D_context, n_estimators=8, device="cuda"):
    """Fit TabPFN as the predictive distribution model for Y | X."""
    fm = TabPFNExtractor(n_estimators=n_estimators, device=device)
    fm.fit(D_context[["X"]].values, D_context["Y"].values)
    return fm


def tabpfn_quantile_samples(fm, X_new, M, rng=None, grid=None):
    """Approximate Y ~ p_phi(. | x, D) by sampling from TabPFN quantiles."""
    if rng is None:
        rng = np.random.default_rng()
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)

    X_new = np.asarray(X_new).reshape(-1, 1)
    q = fm.predict_quantiles(X_new, grid)  # shape: n_points x n_grid

    u = rng.uniform(grid[0], grid[-1], size=(len(X_new), M))
    return np.array([
        np.interp(u_i, grid, q_i)
        for u_i, q_i in zip(u, q)
    ])


def sample_y_given_x_tabpfn(fm, x_vec, S, rng=None, q_grid=None):
    """Draw an n x S matrix of Monte Carlo samples from TabPFN's quantile model."""
    return tabpfn_quantile_samples(
        fm,
        np.asarray(x_vec).reshape(-1, 1),
        M=S,
        rng=rng,
        grid=q_grid,
    )


def estimate_R_route1_mc(fm, x_vec, lambda_grid, mu_hat_fn, S=500,
                         batch_size=256, rng=None, q_grid=None):
    """
    Estimate R(lambda | x) by Route 1 Monte Carlo sampling from fitted Y | X.

    Returns an n x L matrix. Each row is the estimated risk curve for one x.
    """
    if rng is None:
        rng = np.random.default_rng()
    if q_grid is None:
        q_grid = np.linspace(0.01, 0.99, 99)

    x_vec = np.asarray(x_vec)
    n = len(x_vec)
    L = len(lambda_grid)
    R_hat = np.empty((n, L))

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        x_batch = x_vec[start:end]

        y_samples = sample_y_given_x_tabpfn(
            fm,
            x_batch,
            S=S,
            rng=rng,
            q_grid=q_grid
        )
        mu_hat_batch = mu_hat_fn(x_batch)
        residual_samples = np.abs(y_samples - mu_hat_batch[:, None])

        losses = bounded_excess_loss(
            residual_samples[:, :, None],
            lambda_grid[None, None, :]
        )
        R_hat[start:end] = losses.mean(axis=1)

    return np.clip(R_hat, 0.0, B_LOSS)


## 5. Invert risk curves and calibrate CRC


In [ ]:
def inverter_R(R_row, lambda_grid, a):
    """inf { lambda in grid : R(lambda) <= a }."""
    mask = R_row <= a
    if not mask.any():
        return float(lambda_grid[-1])
    return float(lambda_grid[np.argmax(mask)])


def calibrar_crc(losses_cal, lambda_grid, alpha, B=B_LOSS):
    """Standard CRC: smallest lambda with corrected calibration risk <= alpha."""
    n = losses_cal.shape[0]
    R_cal = losses_cal.mean(axis=0)
    R_upper = (n * R_cal + B) / (n + 1)
    mask = R_upper <= alpha
    if not mask.any():
        return float(lambda_grid[-1])
    return float(lambda_grid[np.argmax(mask)])


def calibrar_crc_rectified(losses_rect, a_grid, alpha, B=B_LOSS):
    """Rectified CRC: largest risk budget a with corrected risk <= alpha."""
    n = losses_rect.shape[0]
    R_cal = losses_rect.mean(axis=0)
    R_upper = (n * R_cal + B) / (n + 1)
    mask = R_upper <= alpha
    if not mask.any():
        return float(a_grid[0])
    return float(a_grid[np.where(mask)[0].max()])


## 6. One-replication pipeline


In [ ]:
@dataclass
class ResultadoRodada:
    df_metricas: pd.DataFrame
    a_hat_oracle: float
    a_hat_route1: float
    lambda_fixo: float
    bandas: dict


def rodar_uma_rodada(n_train, n_context, n_cal, n_test, alpha, L=41,
                     S_mc=500, B=B_LOSS, device="cuda", n_estimators=8, verbose=False, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    lambda_grid = np.linspace(0, 4, L)
    a_grid = np.linspace(0.001, 1, 40)

    D_train = gerar_dataset(n_train, rng)
    D_context = gerar_dataset(n_context, rng)
    cal = gerar_dataset(n_cal, rng)
    test = gerar_dataset(n_test, rng)

    if verbose:
        print("  fitting base mean model on training sample")
    gam_mu = fit_mu(D_train)
    mu_hat_fn = lambda x: predict_mu(gam_mu, x)

    mu_cal = mu_hat_fn(cal["X"].values)
    mu_test = mu_hat_fn(test["X"].values)

    res_cal = np.abs(cal["Y"].values - mu_cal)
    res_test = np.abs(test["Y"].values - mu_test)

    # ====== M1) Standard CRC ======
    if verbose:
        print("  M1 Standard CRC")
    losses_cal_fixo = bounded_excess_loss(
        res_cal[:, None],
        lambda_grid[None, :]
    )
    lambda_fixo = calibrar_crc(losses_cal_fixo, lambda_grid, alpha, B=B)
    loss_fixo = bounded_excess_loss(res_test, lambda_fixo)
    width_fixo = np.full(n_test, 2 * lambda_fixo)

    def rectificar(R_cal_mat, R_test_mat):
        losses_cal_rect = np.empty((n_cal, len(a_grid)))

        for j, a in enumerate(a_grid):
            for i in range(n_cal):
                lam_i = inverter_R(R_cal_mat[i], lambda_grid, a)
                losses_cal_rect[i, j] = bounded_excess_loss(res_cal[i], lam_i)

        a_hat = calibrar_crc_rectified(losses_cal_rect, a_grid, alpha, B=B)

        loss_t = np.empty(n_test)
        width_t = np.empty(n_test)
        lambda_t = np.empty(n_test)

        for i in range(n_test):
            lam_i = inverter_R(R_test_mat[i], lambda_grid, a_hat)
            lambda_t[i] = lam_i
            loss_t[i] = bounded_excess_loss(res_test[i], lam_i)
            width_t[i] = 2 * lam_i

        return a_hat, loss_t, width_t, lambda_t

    # ====== M2) Oracle Rectified CRC ======
    if verbose:
        print("  M2 Oracle Rectified CRC")
    R_or_cal = oracle_R_matriz(cal["X"].values, lambda_grid, mu_hat_fn)
    R_or_test = oracle_R_matriz(test["X"].values, lambda_grid, mu_hat_fn)
    a_hat_or, loss_or, width_or, lambda_or = rectificar(R_or_cal, R_or_test)

    # ====== M3) Rectified Route 1 via estimated Y | X ======
    if verbose:
        print("  M3 Rectified Route 1: fitting TabPFN Y|X quantile sampler")
    t0 = time.time()
    y_sampler = fit_y_given_x_sampler_tabpfn(
        D_context,
        n_estimators=n_estimators,
        device=device
    )
    if verbose:
        print(f"     sampler fit time: {time.time() - t0:.1f}s")

    if verbose:
        print(f"     estimating Monte Carlo risk matrices with S={S_mc}")
    t0 = time.time()
    R_r1_cal = estimate_R_route1_mc(
        y_sampler,
        cal["X"].values,
        lambda_grid,
        mu_hat_fn,
        S=S_mc,
        rng=rng
    )
    if verbose:
        print(f"     calibration MC time: {time.time() - t0:.1f}s")

    t0 = time.time()
    R_r1_test = estimate_R_route1_mc(
        y_sampler,
        test["X"].values,
        lambda_grid,
        mu_hat_fn,
        S=S_mc,
        rng=rng
    )
    if verbose:
        print(f"     test MC time: {time.time() - t0:.1f}s")

    a_hat_r1, loss_r1, width_r1, lambda_r1 = rectificar(R_r1_cal, R_r1_test)

    # ====== Metrics by difficulty bin ======
    G = 5
    bin_edges = np.quantile(test["sigma_true"].values, np.linspace(0, 1, G + 1))
    bin_edges[0] -= 1e-9
    bins = np.searchsorted(bin_edges[1:-1], test["sigma_true"].values, side="right")

    def metricas(loss, width, lambda_vec, metodo):
        R_bins = np.array([
            loss[bins == g].mean() if (bins == g).any() else np.nan
            for g in range(G)
        ])
        return {
            "metodo": metodo,
            "R_test": loss.mean(),
            "cobertura_emp": (loss == 0).mean(),
            "max_bin_risk": np.nanmax(R_bins),
            "media_excesso_bin": np.nanmean(np.maximum(R_bins - alpha, 0)),
            "width_medio": width.mean(),
            "lambda_sd": lambda_vec.std(ddof=1) if len(lambda_vec) > 1 else 0.0,
            "R_bins": R_bins.tolist(),
        }

    df_metricas = pd.DataFrame([
        metricas(loss_fixo, width_fixo, np.full(n_test, lambda_fixo), "crc-padrao"),
        metricas(loss_or, width_or, lambda_or, "oracle-rect"),
        metricas(loss_r1, width_r1, lambda_r1, "rect-route1"),
    ])

    bandas = {
        "crc-padrao": dict(
            lower=mu_test - lambda_fixo,
            upper=mu_test + lambda_fixo,
            X=test["X"].values,
            Y=test["Y"].values,
        ),
        "oracle-rect": dict(
            lower=mu_test - lambda_or,
            upper=mu_test + lambda_or,
            X=test["X"].values,
            Y=test["Y"].values,
        ),
        "rect-route1": dict(
            lower=mu_test - lambda_r1,
            upper=mu_test + lambda_r1,
            X=test["X"].values,
            Y=test["Y"].values,
        ),
    }

    return ResultadoRodada(df_metricas, a_hat_or, a_hat_r1, lambda_fixo, bandas)


## 7. Run the Monte Carlo experiment

`S_mc` controls the number of Route 1 Monte Carlo samples per evaluated input. Increase it for a final simulation study.


In [ ]:
config = dict(
    n_train=500,
    n_context=500,
    n_cal=300,
    n_test=1000,
    alpha=0.10,
    B=B_LOSS,
    L=41,
    S_mc=500,
    n_estimators=8,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

n_rodadas = 20

todos_resultados = []
bandas_r1 = None

for r in range(n_rodadas):
    print(f"\n--- Monte Carlo replication {r + 1}/{n_rodadas} ---")
    t_ini = time.time()
    rng_r = np.random.default_rng(2026 + r + 1)

    out = rodar_uma_rodada(**config, verbose=(r == 0), rng=rng_r)

    print(
        f"  time: {time.time() - t_ini:.1f}s | "
        f"lambda_standard={out.lambda_fixo:.3f} | "
        f"a_hat_oracle={out.a_hat_oracle:.3f} | "
        f"a_hat_route1={out.a_hat_route1:.3f}"
    )

    df_r = out.df_metricas.copy()
    df_r["rodada"] = r + 1
    todos_resultados.append(df_r)

    if r == 0:
        bandas_r1 = out.bandas


In [ ]:
todos = pd.concat(todos_resultados, ignore_index=True)

resumo = (todos
    .drop(columns=["R_bins"])
    .groupby("metodo")
    .agg(["mean", "std"])
    .round(4))
print("\n========== RESUMO ==========")
print(resumo)


## 8. Visualizations


In [ ]:
method_order = ["crc-padrao", "rect-route1", "oracle-rect"]
method_labels = {
    "crc-padrao": "Standard CRC",
    "rect-route1": "Rectified Route 1",
    "oracle-rect": "Oracle Rectified",
}
method_colors = {
    "crc-padrao": "#999999",
    "rect-route1": "#0072B2",
    "oracle-rect": "#D55E00",
}


def add_stripplot_metric(ax, variable, title, ylabel, horizontal_line=None):
    sns.stripplot(
        data=todos,
        x="metodo",
        y=variable,
        order=method_order,
        palette=method_colors,
        ax=ax,
        alpha=0.7,
        size=7,
        jitter=0.1,
    )

    means = todos.groupby("metodo")[variable].mean()
    for i, method in enumerate(method_order):
        ax.hlines(means[method], i - 0.25, i + 0.25, colors="black", linewidth=2)

    if horizontal_line is not None:
        ax.axhline(horizontal_line, linestyle="--", color="gray", alpha=0.6)

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_xticklabels([method_labels[m] for m in method_order], rotation=20)


fig, axes = plt.subplots(2, 2, figsize=(12, 9))

add_stripplot_metric(
    axes[0, 0],
    "R_test",
    "(a) Marginal risk",
    r"$\widehat R_{\mathrm{test}}$",
    horizontal_line=config["alpha"],
)
add_stripplot_metric(
    axes[0, 1],
    "media_excesso_bin",
    "(b) Mean excess by bin",
    "Mean excess",
)
add_stripplot_metric(
    axes[1, 0],
    "max_bin_risk",
    "(c) Worst-bin risk",
    r"$\max_g \widehat R_g$",
    horizontal_line=config["alpha"],
)
add_stripplot_metric(
    axes[1, 1],
    "width_medio",
    "(d) Mean band width",
    "Band width",
)

fig.suptitle(
    f"Standard CRC vs Oracle Rectified vs Rectified Route 1 (GAM + TabPFN quantile sampler)  |  "
    f"alpha={config['alpha']}  |  {n_rodadas} Monte Carlo replications",
    fontweight="bold",
)
fig.tight_layout()
plt.savefig("exp_1_route1_summary_metrics.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
df_bins = todos.copy()
df_bins["R_bins"] = df_bins["R_bins"].apply(
    lambda x: x if isinstance(x, list) else x.tolist()
)
df_bins = df_bins.explode("R_bins")
df_bins["difficulty_bin"] = (
    df_bins.groupby(["rodada", "metodo"]).cumcount() + 1
)
df_bins["R_g"] = pd.to_numeric(df_bins["R_bins"])
df_bins["metodo"] = pd.Categorical(df_bins["metodo"], categories=method_order)

df_bins_summary = (
    df_bins
    .groupby(["metodo", "difficulty_bin"], observed=False)["R_g"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
df_bins_summary["se"] = df_bins_summary["std"] / np.sqrt(df_bins_summary["count"])

fig, ax = plt.subplots(figsize=(9, 6))

for method in method_order:
    data_m = df_bins_summary[df_bins_summary["metodo"] == method]
    ax.plot(
        data_m["difficulty_bin"],
        data_m["mean"],
        "o-",
        color=method_colors[method],
        label=method_labels[method],
        linewidth=2,
        markersize=6,
    )
    ax.fill_between(
        data_m["difficulty_bin"],
        data_m["mean"] - 2 * data_m["se"],
        data_m["mean"] + 2 * data_m["se"],
        color=method_colors[method],
        alpha=0.15,
    )

ax.axhline(config["alpha"], linestyle="--", color="gray", alpha=0.6)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(["1\n(easy)", "2", "3", "4", "5\n(hard)"])
ax.set_xlabel("Difficulty bin")
ax.set_ylabel(r"$\widehat R_g$")
ax.set_title(
    "Conditional risk by difficulty bin\n"
    r"Bins are defined by quantiles of the true $\sigma(x)$"
)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig("exp_1_route1_conditional_risk_by_difficulty_bin.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d


def smooth_width_for_plot(X, width, grid_x, smooth_sigma=10.0):
    order_idx = np.argsort(X)
    X_sorted = X[order_idx]
    width_sorted = width[order_idx]
    width_fun = interp1d(
        X_sorted,
        width_sorted,
        bounds_error=False,
        fill_value="extrapolate",
    )
    width_grid = width_fun(grid_x)
    width_grid = gaussian_filter1d(width_grid, sigma=smooth_sigma)
    return np.maximum(width_grid, 1e-8)


fig, ax = plt.subplots(figsize=(8.8, 5.0))
grid_x = np.linspace(-2, 2, 600)

for method in method_order:
    band_data = bandas_r1[method]
    width = band_data["upper"] - band_data["lower"]
    width_grid = smooth_width_for_plot(
        X=band_data["X"],
        width=width,
        grid_x=grid_x,
        smooth_sigma=10.0,
    )
    ax.plot(
        grid_x,
        width_grid,
        color=method_colors.get(method, "steelblue"),
        linewidth=2.4,
        label=method_labels.get(method, method),
    )

sigma_grid = sigma_true(grid_x)
sigma_scaled = sigma_grid / sigma_grid.max()

ax2 = ax.twinx()
ax2.plot(
    grid_x,
    sigma_scaled,
    color="black",
    linestyle="--",
    linewidth=1.6,
    alpha=0.65,
    label=r"Scaled true $\sigma(x)$",
)

ax.set_xlabel(r"$X$")
ax.set_ylabel(r"Band width $2\lambda(x)$")
ax2.set_ylabel(r"Scaled true $\sigma(x)$")
ax.set_title("Band width as a function of covariate difficulty")

lines_1, labels_1 = ax.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    frameon=False,
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, -0.16),
)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig("exp_1_route1_band_width_by_x.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# Prediction bands colored by bounded absolute excess loss
# Smoothed bands only for visualization
#
# Important: all risks and losses are computed with the original,
# unsmoothed bands. The smoothing below is used only to make the
# shaded bands easier to read in the figure.
# ============================================================

from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d


# ------------------------------------------------------------
# Loss from stored bands
# ------------------------------------------------------------

def bounded_absolute_excess_loss_from_band(band_data, excess_scale=EXCESS_SCALE):
    """
    Computes the bounded absolute excess loss from stored prediction bands.

    The stored bands are assumed to satisfy:
        lower = mu_hat - lambda
        upper = mu_hat + lambda
    """
    X = band_data["X"]
    Y = band_data["Y"]
    lower = band_data["lower"]
    upper = band_data["upper"]

    mu_hat = 0.5 * (lower + upper)
    lam = 0.5 * (upper - lower)
    residual = np.abs(Y - mu_hat)

    loss = np.minimum(
        1.0,
        np.maximum(0.0, residual - lam) / excess_scale
    )
    width = upper - lower

    return loss, width, mu_hat, lam


# ------------------------------------------------------------
# Smooth prediction bands for visualization only
# ------------------------------------------------------------

def smooth_band_for_plot(X, lower, upper, grid_x, smooth_sigma=8.0):
    """
    Smooth prediction bands only for visualization.

    The smoothing is applied separately to the band center and width.
    Risk and loss calculations still use the original, unsmoothed bands.
    """
    order_idx = np.argsort(X)

    X_sorted = X[order_idx]
    lower_sorted = lower[order_idx]
    upper_sorted = upper[order_idx]

    center_sorted = 0.5 * (lower_sorted + upper_sorted)
    width_sorted = upper_sorted - lower_sorted

    center_fun = interp1d(
        X_sorted,
        center_sorted,
        bounds_error=False,
        fill_value="extrapolate"
    )
    width_fun = interp1d(
        X_sorted,
        width_sorted,
        bounds_error=False,
        fill_value="extrapolate"
    )

    center_grid = center_fun(grid_x)
    width_grid = width_fun(grid_x)

    center_grid = gaussian_filter1d(center_grid, sigma=smooth_sigma)
    width_grid = gaussian_filter1d(width_grid, sigma=smooth_sigma)
    width_grid = np.maximum(width_grid, 1e-8)

    lower_grid = center_grid - 0.5 * width_grid
    upper_grid = center_grid + 0.5 * width_grid

    return lower_grid, upper_grid


# ============================================================
# Plot
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.8),
    sharey=True,
    constrained_layout=True
)

grid_x = np.linspace(-2, 2, 600)
mu_grid = mu_true(grid_x)

scatter_handle = None

for ax, method in zip(axes, method_order):
    band_data = bandas_r1[method]

    X = band_data["X"]
    Y = band_data["Y"]
    lower = band_data["lower"]
    upper = band_data["upper"]

    loss_i, width_i, mu_hat_i, lambda_i = bounded_absolute_excess_loss_from_band(
        band_data,
        excess_scale=EXCESS_SCALE
    )

    lower_grid, upper_grid = smooth_band_for_plot(
        X=X,
        lower=lower,
        upper=upper,
        grid_x=grid_x,
        smooth_sigma=8.0
    )

    ax.fill_between(
        grid_x,
        lower_grid,
        upper_grid,
        color=method_colors.get(method, "steelblue"),
        alpha=0.22,
        linewidth=0
    )

    scatter_handle = ax.scatter(
        X,
        Y,
        c=loss_i,
        s=7,
        alpha=0.55,
        cmap="viridis",
        vmin=0,
        vmax=1,
        edgecolor="none"
    )

    ax.plot(
        grid_x,
        mu_grid,
        color="black",
        linewidth=1.6,
        label=r"True mean $\mu(x)$"
    )

    ax.set_title(method_labels.get(method, method), fontsize=12)
    ax.set_xlabel(r"$X$")
    ax.grid(True, alpha=0.35)

axes[0].set_ylabel(r"$Y$")

colorbar = fig.colorbar(
    scatter_handle,
    ax=axes,
    location="right",
    shrink=0.88,
    pad=0.02
)
colorbar.set_label("Bounded absolute excess loss")

fig.suptitle(
    "Prediction bands colored by bounded absolute excess loss (Route 1 experiment)",
    fontsize=15,
    fontweight="bold"
)

plt.savefig(
    "exp_1_route1_prediction_bands_absolute_loss_refined.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()
